# CSE 25 - Introduction to Artificial Intelligence
## Week 5 Thursday: Gradient descent and backpropagation

**Learning Objectives:**
- Describe the forward computation from inputs to loss  
- Explain the role of a loss function in learning
- Interpret a computation graph for a simple model  
- Explain why backpropagation is more efficient than finite differences
- Apply the chain rule to compute gradients in a computation graph
- Implement backpropagation for a sigmoid + binary cross-entropy model
- Use gradients to update model parameters and reduce loss


Instructions

Use your copy of this notebook on Datahub and complete it during class. Work through the cells below **in order**. You may discuss with your neighbors, but make sure you understand each step yourself.


SUBMISSION:
When finished, download it as a PDF and upload it to Gradescope under `In-Class – Week 5 Thursday` to receive credit. To download it as a PDF, on DataHub go to `File -> Save and Export Notebook As -> PDF`.

#### Indexing Convention

A note on notation: we're using **two different indices** to distinguish between the feature index and the example index: 

- **Feature index**: $i = 1, \dots, n$  
  Refers to *which input feature* ($x_1, x_2, \dots$).

- **Example index**: $j = 0, \dots, N-1$  
  Refers to *which training example* in the dataset.


In symbols,
$$
x_i^{(j)} \quad = \qquad \text{feature } i \text{ of training example } j
$$

In Python, indexing starts at **0**, so this corresponds to:

$$
x_i^{(j)}\quad = \qquad  \texttt{X[j][i-1]}
$$

*Some libraries add a dummy feature $x_0$ set to $1$ that corresponds to the bias to fix the off-by-1 indexing.*


Last time: given model parameters $w_i$ and $b$, to predict the label of an input example and evaluate the prediction:

1. Compute a score
   $$z = \sum_{i=1}^n w_i x_i + b$$

2. Turn the score into a confidence using the sigmoid function
   $$p = \sigma(z)$$

3. Measure how good that confidence was using a loss (given the true label $y$)
   $$\mathcal{L}(p, y) = -\big(y \log(p) + (1-y)\log(1-p)\big)$$

We call this the *forward* computation:
  
$$x \xrightarrow{\,w,b\,} z \xrightarrow{\,\sigma(.)\,} p \xrightarrow{\,y\,} \mathcal{L}_{\text{train}}$$

Each arrow means "is computed from" or "depends on". Equivalently, the loss *depends on* $p$, which *depends on* $z$, which *depends on* $w$ and $b$.

This sequence is the **computation graph** for our model.

In [11]:
import math

def stable_sigmoid(z):
    '''
    This implementation avoids overflow issues by handling large positive and negative values of z separately.

    z: a numeric value.

    if z >= 0: use the 'standard' formula: 1/(1 + exp(-z))
    if z < 0: use the alternative formula to avoid overflow: exp(z) / (1 + exp(z))
    '''
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    else:
        ez = math.exp(z)
        return ez / (1 + ez)

def stable_binary_cross_entropy(p, y):
    '''
    Clips p to avoid (overflow) issues, then computes binary cross-entropy loss.
    
    p: model confidence (between 0 and 1)
    y: true label (0 or 1)
    '''
    eps = 1e-8

    if p < eps:
        p = eps
    if p > 1 - eps:
        p = 1 - eps

    return -(y * math.log(p) + (1 - y) * math.log(1 - p))

In gradient descent for linear regression, we reduced error by asking:

- If we change a parameter slightly, how does the error change?
- In which direction should we move the parameter to reduce error?
- How sensitive is the error to that parameter?

We answered these questions by treating error as a function of the parameters and using **gradients** to understand how changes in the parameters affect the error.

Here, with logistic regression, we use the same idea, but with **loss**.

The goal is the same: understand how the loss would change with respect to the parameters so that we can update $w$ and $b$ to reduce loss.

Our model has three parameters:
- `w1`: weight for feature x1
- `w2`: weight for feature x2
- `b`: bias

Computation Graph: 

$x \xrightarrow{\,w,b\,} z \xrightarrow{\,\sigma(.)\,} p \xrightarrow{\,y\,} L$

In [12]:
# The forward pass to get confidence values in predictions for training examples based on model parameters
def forward(X, w1, w2, b):
    '''
    Inputs: 
    X: list of data points, where each point is a list of 2 features [x1, x2]
    w1, w2: weights for the two features
    b: bias term

    Output: A list of p (confidence values between 0 and 1) for each data point
    '''
    
    # Initialize an empty list to store confidence values
    p_list = []

    # Iterate through each data point in X
    for j in range(len(X)):
        x1 = X[j][0]
        x2 = X[j][1]
        
        # Calculate the linear score z = w1*x1 + w2*x2 + b
        z = w1 * x1 + w2 * x2 + b
        
        # Use the score to get the sigmoid confidence
        p = stable_sigmoid(z)

        # Append the confidence value to the list
        p_list.append(p)
        
    return p_list

In [13]:
# The loss function, given a list of confidence values and true labels
def get_avg_binary_cross_entropy(p_list, y):
    '''
    p_list: list of predicted confidence values (between 0 and 1)
    y: list of true labels (0 or 1)
    '''
    total_loss = 0.0
    for j in range(len(p_list)):
        p = p_list[j]
        y_true = y[j]
        
        # Safety: Clip p to avoid log(0) error
        loss_j = stable_binary_cross_entropy(p, y_true)
        total_loss += loss_j
        
    return total_loss / len(p_list)

### Calculating the Gradient with Respect to Multiple Parameters

When our model has $n$ input features, the score 
$$
z = \sum_{i=1}^n w_ix_i + b
$$
so the model prediction (and hence also the loss) depends on all the parameters: the weights ($w_1, w_2, \dots, w_n$) and a bias $b$. 
For concreteness, we'll work with the two feature case:
$$
y = w_1 x_1 + w_2 x_2 + b
$$
where the loss depends on $w_1$, $w_2$, and $b$. 

The loss $\mathcal{L}(p, y)$ measures how well the model's prediction $p$ matches the true label $y$ for a single data point. We cannot change $y$, so we focus on how changes in the prediction confidence $p$ affect the loss.

The **dataset-level loss** is the average loss over all $N$ training examples. Since $p$ depends on $z$, which in turn depends on the parameters of our model, the dataset-level loss is a function of the model parameters:

$$
\mathcal{L}_{\text{train}}(w_1, w_2, b)
= \frac{1}{N} \sum_{j=1}^N \mathcal{L}\!\left(p^{(j)}, y^{(j)}\right),
$$

where
$$
p^{(j)} = \sigma\!\left(z^{(j)}\right),
\qquad
z^{(j)} = w_1 x_1^{(j)} + w_2 x_2^{(j)} + b.
$$

 When we change a parameter (for example, $w_1$), every prediction $p^{(j)}$ may change. As a result, the loss for each data point may change. The partial derivative of the dataset-level loss with respect to this parameter captures the effect of that parameter on the average loss across the dataset. Like before, we can use this effect to look for the minimal loss by changing the parameter in the direction that will reduce the loss.

To look for optimal model parameters, we compute the **partial derivatives** of the average loss with respect to each parameter:

- $\frac{\partial \mathcal{L}_{\text{train}}}{\partial w_1}$: how the average loss changes as $w_1$ changes, holding the other parameters fixed.
- $\frac{\partial \mathcal{L}_{\text{train}}}{\partial w_2}$: how the average loss changes as $w_2$ changes, holding the other parameters fixed.
- $\frac{\partial \mathcal{L}_{\text{train}}}{\partial b}$: how the average loss changes as $b$ changes.

One way to approximate these derivatives is the **finite difference method**, which measures how the average dataset loss changes when we slightly nudge one parameter:

$$
\frac{\partial \mathcal{L}_{\text{train}}}{\partial w_1}
\approx
\frac{\mathcal{L}_{\text{train}}(w_1+\epsilon, w_2, b) - \mathcal{L}_{\text{train}}(w_1, w_2, b)}{\epsilon}
$$

$$
\frac{\partial \mathcal{L}_{\text{train}}}{\partial w_2}
\approx
\frac{\mathcal{L}_{\text{train}}(w_1, w_2+\epsilon, b) - \mathcal{L}_{\text{train}}(w_1, w_2, b)}{\epsilon}
$$

$$
\frac{\partial \mathcal{L}_{\text{train}}}{\partial b}
\approx
\frac{\mathcal{L}_{\text{train}}(w_1, w_2, b+\epsilon) - \mathcal{L}_{\text{train}}(w_1, w_2, b)}{\epsilon}
$$

Here, $\epsilon$ is a small number. This idea works for any number of parameters: we nudge one parameter at a time and observe how the **average loss over the dataset** changes. 

In [14]:
# The gradient calculation using the finite difference method
# To see how (w,b) affect average loss, we nudge one parameter at a time and look at how that affects the loss.

# Remember: "Run Forward" means calculate model score for each example and then 
# apply sigmoid function to get list of confidence values for each example.

# eps controls the size of the parameter nudge.
# Too small: floating-point rounding dominates.
# Too large: the linear approximation breaks down.

def get_gradients(X, y, w1, w2, b):
    eps = 1e-4 # finite-difference step size (larger than clipping threshold)
    
    # STEP 0: Baseline - Where is average loss at now?
    # We must Run Forward -> Then Calculate Loss
    base_conf = forward(X, w1, w2, b)
    base_loss  = get_avg_binary_cross_entropy(base_conf, y)
    
    # STEP 1: Get gradient for w1
    # Nudge w1 -> Re-run Forward -> Re-calculate Loss
    w1_conf = forward(X, w1 + eps, w2, b)
    w1_loss  = get_avg_binary_cross_entropy(w1_conf, y)
    grad_w1  = (w1_loss - base_loss) / eps
    
    # STEP 2: Get gradient for w2
    # Nudge w2 -> Re-run Forward -> Re-calculate Loss
    w2_conf = forward(X, w1, w2 + eps, b)
    w2_loss  = get_avg_binary_cross_entropy(w2_conf, y)
    grad_w2  = (w2_loss - base_loss) / eps
    
    # STEP 3: Get gradient for bias
    # Nudge b -> Re-run Forward -> Re-calculate Loss
    b_conf = forward(X, w1, w2, b + eps)
    b_loss  = get_avg_binary_cross_entropy(b_conf, y)
    grad_b  = (b_loss - base_loss) / eps
     
    return grad_w1, grad_w2, grad_b

Trace through `get_gradients` line by line and count each call to `forward(...)`.
Then look at the `forward` function and count how many times it calls `stable_sigmoid` for a dataset of size $N$.

Q: How many times does the `get_gradients` function run the forward computation?

`YOUR ANSWER HERE`

Q: For a dataset of $N$ examples, how many times does `stable_sigmoid` get called in total? Express your answer in terms of $N$.

`YOUR ANSWER HERE`

Q: If the model had $k$ features instead of two, how would the number of forward passes change? Express your answer in terms of $k$.

`YOUR ANSWER HERE`

### Gradient descent with finite difference approximations

Our training loop minimizes the **average loss over the entire dataset**:

$$
\mathcal{L}_{\text{train}}(w_1, w_2, b) = \frac{1}{N}\sum_{j=1}^N \mathcal{L}(p^{(j)}, y^{(j)})
$$

Using the dataset-level loss for specific parameter values, we can approximate the partial derivative with respect to each parameter.

To calculate the dataset-level loss, we need to run the model to calculate the confidence values for all examples with the given model paramater values.



In [15]:
# Load the toy data again for the next section
X_toy = [
    [1.5, 4],
    [1, 2],
    [2, 1],
    [3, 5],
    [4, 2],
    [0, 0],
    [1.5, -0.5]
]
y_toy = [1, 1, 0, 1, 0, 0, 0]

In [ ]:
# The Training Loop, using finite difference approximations for gradient

# Initialize Parameters
w1 = 0.0
w2 = 0.0
b  = 0.0

# Initialize hyperparameters
learning_rate = 0.1
epochs = 500

curr_loss = get_avg_binary_cross_entropy(forward(X_toy, w1, w2, b), y_toy)

print(f"Initial Loss: {curr_loss:.4f}")

for i in range(epochs):
    
    # 1. Calculate Gradients
    dw1, dw2, db = get_gradients(X_toy, y_toy, w1, w2, b)
    
    # 2. Update Weights
    #    Subtraction means we update in the opposite direction to gradient
    w1 = w1 - learning_rate * dw1
    w2 = w2 - learning_rate * dw2
    b  = b  - learning_rate * db
    
    if i % 50 == 0:
        # Check progress
        curr_preds = forward(X_toy, w1, w2, b)
        curr_loss = get_avg_binary_cross_entropy(curr_preds, y_toy)
        print(f"Iter {i}: w1={w1:.2f}, w2={w2:.2f}, b={b:.2f} | Loss={curr_loss:.4f}")

print("\nFinal Result:")
print("Final loss:", curr_loss)

print(f"w1: {w1:.4f}, w2: {w2:.4f}, b: {b:.4f}")

Initial Loss: 0.6931
Iter 0: w1=-0.01, w2=0.06, b=-0.01 | Loss=0.6560
Iter 50: w1=-0.84, w2=1.07, b=-0.46 | Loss=0.2255
Iter 100: w1=-1.17, w2=1.50, b=-0.72 | Loss=0.1522
Iter 150: w1=-1.37, w2=1.79, b=-0.92 | Loss=0.1196
Iter 200: w1=-1.51, w2=2.01, b=-1.08 | Loss=0.1000
Iter 250: w1=-1.62, w2=2.20, b=-1.22 | Loss=0.0866
Iter 300: w1=-1.71, w2=2.37, b=-1.34 | Loss=0.0766
Iter 350: w1=-1.79, w2=2.51, b=-1.45 | Loss=0.0688
Iter 400: w1=-1.86, w2=2.64, b=-1.55 | Loss=0.0625
Iter 450: w1=-1.92, w2=2.76, b=-1.64 | Loss=0.0573

Final Result:
Final loss: 0.05727086272438568
w1: -1.9785, w2: 2.8661, b: -1.7203


We just trained our model by repeating the following steps:

- Running the model forward to get the confidence values $p$
- Computing the loss using $p$ and $y$
- Nudging one parameter at a time (re-running the model for each) to see how the loss changes
- Updating the parameters using those estimates

This works, but it is slow and doesn't scale well as the training data set gets large and as the number of parameters in the model gets large.

A better algorithm would compute gradients more efficiently without rerunning the model separately for each parameter.

### Backpropagation

Each step in the forward computation depends only on the one before it:

- The pointwise loss $\mathcal{L}$ depends on computed confidence $p$ for the current model parameters and the (fixed) true label $y$ for this example
- The confidence $p$ for this example with these model parameters depends on the model score $z$ for this example
- The model score $z$ depends on the (fixed) example $x$ and the model parameters $w$ and $b$

Instead of changing one parameter at a time and rerunning everything, we can:

1. Start at the loss
2. Measure how sensitive the loss is to its input
3. Pass that sensitivity backward through the computation

This is called **backpropagation**. Informally, propagating the loss back through the chain in the computation graph.

$$x \xrightarrow{\,w,b\,} z \xrightarrow{\,\sigma(.)\,} p \xrightarrow{\,y\,} \mathcal{L}$$

For the computation graph above, backpropagation answers the following questions:

1. If the value of $p$ changes slightly, how much does the loss $\mathcal{L}$ change? In symbols:
$$\frac{\partial \mathcal{L}}{\partial p}$$
2. If the score $z$ changes slightly, how much does the prediction $p$ change? In symbols: 
$$\frac{\partial p}{\partial z}$$
3. If a parameter ($w$ or $b$) changes slightly, how much does the score $z$ change? In symbols
$$\frac{\partial z}{\partial w_i} \qquad \qquad \frac{\partial z}{\partial b}$$

Once we know these **local effects**, we multiply them together.

This is the **chain rule**, applied to a computation graph.

For each parameter $w_i$, the chain rule gives:

$$
\frac{\partial \mathcal{L}}{\partial w_i}=
\frac{\partial \mathcal{L}}{\partial p}
\cdot
\frac{\partial p}{\partial z}
\cdot
\frac{\partial z}{\partial w_i}
$$

Similarly, for the bias:

$$
\frac{\partial \mathcal{L}}{\partial b}=
\frac{\partial \mathcal{L}}{\partial p}
\cdot
\frac{\partial p}{\partial z}
\cdot
\frac{\partial z}{\partial b}
$$

Using some calculus (Math 20A, Math 20C) we can directly derive formulas for these expressions.
You are **not expected** to derive the expressions in this course; we'll jump to the final result because we don't assume you've seen calculus already. But if you're interested in seeing the step-by-step derivation, Prof. Bonjour wrote it out [here](https://drive.google.com/file/d/1XSSyNVk8HMhs4gKi4odK_hX7BBldkYzb/view?usp=sharing).

#### Closed-form expressions for partial derivatives

For **one example** $(x, y)$ with model score $z$ and model confidence $p$, the **gradient of the loss with respect to the score $z$** is:

$$
\frac{\partial \mathcal{L}}{\partial z} = \frac{\partial \mathcal{L}}{\partial p}
\cdot
\frac{\partial p}{\partial z} = p - y
$$



For **one example** $(x, y)$ with model parameters $w_1, \ldots, w_n$ and $b$, the **gradient of the score with respect to each model parameter** is:

$$
\frac{\partial z}{\partial w_i} = x_i
\qquad \qquad
\frac{\partial z}{\partial b} = 1
$$

Putting these together using the chain rule, for **one example** the **gradient of the loss with respect to each model parameter** is:

$$
\frac{\partial \mathcal{L}}{\partial w_i} =
\frac{\partial \mathcal{L}}{\partial p}
\cdot
\frac{\partial p}{\partial z}
\cdot
\frac{\partial z}{\partial w_i}
= (p - y)\,x_i
$$

$$
\frac{\partial \mathcal{L}}{\partial b}=
\frac{\partial \mathcal{L}}{\partial p}
\cdot
\frac{\partial p}{\partial z}
\cdot
\frac{\partial z}{\partial b} = p - y
$$

#### Understanding the signal $p - y$

We'll be using these partial derivatives in the update rule during gradient descent. The update rule $$w_{\text{new}} = w_{\text{old}} - \alpha \cdot \frac{\partial \mathcal{L}}{\partial w}$$ updates parameters by moving in the **opposite**
direction of the gradient. 

We hope that moves the score in the direction that would bring $p$ *closer to $y$*.

Let's try some examples.

1. If $y = 1$ and $p = 0.8$:
   - Is the gradient positive or negative?
   - Will the update push the score $z$ **up** or **down**?

   `YOUR ANSWER HERE`

2. If $y = 0$ and $p = 0.8$:
   - Is gradient positive or negative?
   - Will the update push the score $z$ **up** or **down**?

   `YOUR ANSWER HERE`


**Summarizing**

- If the model predicts **too high** ($p > y$), the gradient is positive -> decrease the score next time.
- If the model predicts **too low** ($p < y$), the gradient is negative -> increase the score next time.
- If the prediction is correct ($p \approx y$), the gradient is near zero.

Notice that the *magnitude* of the update is proportional to the error: large mistakes produce large gradient signals, small mistakes produce small signals.

This property falls out automatically from binary cross-entropy and the sigmoid, without any special design.

### Implementing Backpropagation Gradients

Fill in the `YOUR CODE HERE` sections in the function below to compute the gradients of the average dataset loss with respect to `w1`, `w2`, and `b` using the backpropagation rules for sigmoid + binary cross-entropy.

*Note*: the partial derivative of the average dataset loss is the average of the partial derivatives for each example.

In [ ]:
def get_backprop_grads_dataset(X, y, w1, w2, b):
    """
    Gradients of the AVERAGE dataset loss w.r.t. (w1, w2, b)
    for sigmoid + BCE.

    For one example:
        dL/dw1 = (p - y) * x1
        dL/dw2 = (p - y) * x2
        dL/db  = (p - y)

    X: list of data points, where each point is a list of 2 features [x1, x2]
    y: list of true labels (0 or 1)
    w1, w2: weights for the two features
    b: bias term

    Returns: dL/dw1, dL/dw2, dL/db
    """
    
    grad_w1 = 0.0
    grad_w2 = 0.0
    grad_b  = 0.0

    # Number of examples in training data
    N = len(X)
    
    # Run forward once to get all the confidence values (p) for the dataset
    p_list = forward(X, w1, w2, b)
    
    for j in range(N):
        x1 = X[j][0]
        x2 = X[j][1]
        yj = y[j]
        pj = p_list[j]

        # Gradient of loss w.r.t. z (the linear score)
        # For sigmoid + BCE: dL/dz = p - y
        #dL_dz = ... # YOUR CODE HERE
        dL_dz = pj - yj

        # Gradient of loss w.r.t. parameters (using the simplified backprop rule)
        # Accumulate gradients over the dataset by summing contributions from each example
        grad_w1 += ... # YOUR CODE HERE
        grad_w2 += ... # YOUR CODE HERE
        grad_b  += ... # YOUR CODE HERE


    # Divide SUM of gradients by number of examples to get gradient of AVERAGE loss
    grad_w1 /= N
    grad_w2 /= N
    grad_b  /= N

    return grad_w1, grad_w2, grad_b


Now, let's compare the gradients from backpropagation to the finite-difference method.

In [24]:
# Compare finite-difference gradients vs backprop gradients at some test params
w1_test, w2_test, b_test = 0.3, -0.2, 0.1

fd = get_gradients(X_toy, y_toy, w1_test, w2_test, b_test)
bp = get_backprop_grads_dataset(X_toy, y_toy, w1_test, w2_test, b_test)

print("Finite difference approximation:", fd)
print("Backpropagation:                ", bp)
print("Delta for two methods:          ", (abs(fd[0]-bp[0]), abs(fd[1]-bp[1]), abs(fd[2]-bp[2])))

print("Note: We expect the delta for the two methods to be very small. Is that what we see above?")

Finite difference approximation: (0.3187652925751294, -0.5760339745775056, 0.13618643583623857)
Backpropagation:                 (0.3187096593693251, -0.5761218426325828, 0.13617456901335703)
Delta for two methods:           (5.563320580431741e-05, 8.786805507721152e-05, 1.1866822881539951e-05)
Note: We expect the delta for the two methods to be very small. Is that what we see above?


### Gradient descent with backpropagation

Our training loop minimizes the **average loss over the entire dataset**:

$$
\mathcal{L}_{\text{train}}(w_1, w_2, b) = \frac{1}{N}\sum_{j=1}^N \mathcal{L}(p^{(j)}, y^{(j)})
$$

where

$$
p_j = \sigma(z_j) \qquad \qquad z_j = w_1 x_{1}^{(j)} + w_2 x_{2}^{(j)} + b
$$

So the gradients we want are:

$$
\frac{\partial \mathcal{L}_{\text{train}}}{\partial w_1}\qquad\qquad
\frac{\partial \mathcal{L}_{\text{train}}}{\partial w_2}\qquad\qquad
\frac{\partial \mathcal{L}_{\text{train}}}{\partial b}
$$

Using the chain rule (aka backpropagation) we can use the formulas for the sigmoid function and binary cross entropy to express the partial derivative of the dataset-level loss as a simple function of the current model predictions.

Note that, backprop still needs the forward predictions $p$. The speedup is that we no longer rerun the model once per parameter nudge.
Using the dataset-level loss for specific parameter values, we can approximate the partial derivative with respect to each parameter.




In [20]:
# The Training Loop, using backprop

# Initialize Parameters
w1 = 0.0
w2 = 0.0
b  = 0.0

# Initialize hyperparameters
learning_rate = 0.1
epochs = 500

curr_loss = get_avg_binary_cross_entropy(forward(X_toy, w1, w2, b), y_toy)

print(f"Initial Loss: {curr_loss:.4f}")

for i in range(epochs):
    
    dw1, dw2, db = get_backprop_grads_dataset(X_toy, y_toy, w1, w2, b)
    
    w1 = w1 - learning_rate * dw1
    w2 = w2 - learning_rate * dw2
    b  = b  - learning_rate * db
    
    if i % 50 == 0:
        # Check progress
        curr_preds = forward(X_toy, w1, w2, b)
        curr_loss = get_avg_binary_cross_entropy(curr_preds, y_toy)
        print(f"Iter {i}: w1={w1:.2f}, w2={w2:.2f}, b={b:.2f} | Loss={curr_loss:.4f}")

print("\nFinal Result:")
print("Final loss:", curr_loss)

print(f"w1: {w1:.4f}, w2: {w2:.4f}, b: {b:.4f}")

Initial Loss: 0.6931
Iter 0: w1=-0.01, w2=0.06, b=-0.01 | Loss=0.6560
Iter 50: w1=-0.84, w2=1.07, b=-0.46 | Loss=0.2255
Iter 100: w1=-1.17, w2=1.50, b=-0.72 | Loss=0.1522
Iter 150: w1=-1.37, w2=1.79, b=-0.92 | Loss=0.1196
Iter 200: w1=-1.51, w2=2.01, b=-1.08 | Loss=0.1000
Iter 250: w1=-1.62, w2=2.20, b=-1.22 | Loss=0.0866
Iter 300: w1=-1.71, w2=2.37, b=-1.34 | Loss=0.0766
Iter 350: w1=-1.79, w2=2.51, b=-1.45 | Loss=0.0688
Iter 400: w1=-1.86, w2=2.64, b=-1.55 | Loss=0.0625
Iter 450: w1=-1.92, w2=2.76, b=-1.64 | Loss=0.0573

Final Result:
Final loss: 0.057269993386859445
w1: -1.9784, w2: 2.8661, b: -1.7204


### Comparing the two gradient methods

| | Finite Differences | Backpropagation |
|---|---|---|
| Forward passes per update | $k + 2$ (one per feature + bias + baseline) | $1$ |
| Sigmoid calls per update | $(k + 2) \times N$ | $N$ |
| Exact or approximate? | Approximate ($\varepsilon$ error) | Exact (up to floating-point precision) |
| Scales with number of parameters? | No because cost grows with $k$ | Yes because cost is independent of $k$ |

For our toy model with $k = 3$ parameters and $N = 7$ examples, the difference is small.

A modern neural network may have *billions* of parameters. Backpropagation makes training those networks tractable; finite differences do not.
